# Summer Analytics 2026 - Week 2 Hackathon
## E-Commerce Conversion Prediction Challenge

**Goal**: Predict whether a user converts (binary classification)  
**Metric**: F1 Score  
**Approach**: 
1. Combine `train` + `public_test` for maximum training data
2. Feature engineering (ratios, interactions, binned features)
3. **Ensemble**: Weighted average of tuned XGBoost and balanced Random Forest
4. Threshold optimization to maximize F1 score

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

### Step 2: Load Data

In [ ]:
train = pd.read_csv("train.csv")
public_test = pd.read_csv("public_test.csv")
private_test_raw = pd.read_csv("private_test.csv")

print("Train Shape:", train.shape)
print("Public Test Shape:", public_test.shape)
print("Private Test Shape:", private_test_raw.shape)

# Combine train and public_test for more training data
full_train = pd.concat([train, public_test], ignore_index=True)
print("\nCombined Training Data:", full_train.shape)
print("Conversion Rate:", round(full_train['Converted'].mean(), 4))

### Step 3: Exploratory Data Analysis

In [ ]:
print("=== Missing Values ===")
print(full_train.isnull().sum())
print("\n=== Target Distribution ===")
print(full_train['Converted'].value_counts())
print("\n=== Data Types ===")
print(full_train.dtypes)
print("\n=== Categorical Unique Values ===")
print("Device_Type:", full_train['Device_Type'].unique())
print("Traffic_Source:", full_train['Traffic_Source'].unique())

### Step 4: Feature Engineering

In [ ]:
def engineer_features(df):
    """Create new features from existing columns."""
    data = df.copy()
    
    # Ratio features
    data['Pages_Per_Product'] = data['Pages_Viewed'] / data['Products_Viewed'].clip(lower=1)
    data['Time_Per_Page'] = data['Time_On_Site'] / data['Pages_Viewed'].clip(lower=1)
    data['Time_Per_Product'] = data['Time_On_Site'] / data['Products_Viewed'].clip(lower=1)
    
    # Interaction features
    data['Engagement_Score'] = data['Pages_Viewed'] * data['Products_Viewed']
    data['Product_Page_Diff'] = data['Products_Viewed'] - data['Pages_Viewed']
    data['Browsing_Intensity'] = data['Pages_Viewed'] + data['Products_Viewed']
    
    # Income feature
    data['Income_Per_Age'] = data['Income'] / data['Age'].clip(lower=1)
    
    # Binary indicators
    data['High_Pages'] = (data['Pages_Viewed'] >= 20).astype(int)
    data['High_Products'] = (data['Products_Viewed'] >= 20).astype(int)
    data['Long_Session'] = (data['Time_On_Site'] > 15).astype(int)
    data['Is_Returning'] = (data['Previous_Purchases'] > 0).astype(int)
    data['Many_Purchases'] = (data['Previous_Purchases'] >= 4).astype(int)
    
    # Pages * Discount interaction
    data['Pages_Discount'] = data['Pages_Viewed'] * data['Discount_Seen']
    
    # Age bins
    data['Age_Bin'] = pd.cut(data['Age'], bins=[0, 25, 35, 45, 55, 100], labels=False)
    
    return data

full_train = engineer_features(full_train)
private_test = engineer_features(private_test_raw)

print("Features after engineering:", len(full_train.columns))
print("New features:", [c for c in full_train.columns if c not in train.columns])

### Step 5: Preprocessing

In [ ]:
TARGET = "Converted"

# Encode categorical columns
cat_cols = ['Device_Type', 'Traffic_Source']
for col in cat_cols:
    le = LabelEncoder()
    all_vals = pd.concat([full_train[col], private_test[col]]).astype(str)
    le.fit(all_vals)
    full_train[col] = le.transform(full_train[col].astype(str))
    private_test[col] = le.transform(private_test[col].astype(str))
    print(f"{col} classes: {le.classes_}")

# Separate features and target
y = full_train[TARGET]
X = full_train.drop(columns=["User_ID", TARGET])
X_test = private_test.drop(columns=["User_ID"])

# Fill missing values with median
for col in X.columns:
    med = X[col].median()
    X[col] = X[col].fillna(med)
    X_test[col] = X_test[col].fillna(med)

print(f"\nTraining: {X.shape}, Test: {X_test.shape}")
print(f"Missing: train={X.isnull().sum().sum()}, test={X_test.isnull().sum().sum()}")

### Step 6: Define Models (XGBoost + Random Forest Ensemble)

Our testing shows that a weighted ensemble of a tuned **XGBoost** and a balanced **Random Forest** yields the absolute highest F1 score.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate class imbalance for XGBoost
scale_pos = (y == 0).sum() / (y == 1).sum()

# 1. XGBoost (Tuned)
xgb = XGBClassifier(
    n_estimators=500, 
    max_depth=4, 
    learning_rate=0.05,
    subsample=0.8, 
    colsample_bytree=0.7, 
    min_child_weight=5,
    gamma=0.1, 
    reg_alpha=0.1, 
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    random_state=42, 
    eval_metric='logloss'
)

# 2. Random Forest (Balanced)
rf = RandomForestClassifier(
    n_estimators=500, 
    max_depth=10,
    min_samples_split=15, 
    min_samples_leaf=8,
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

### Step 7: Cross-Validation & Threshold Optimization

We use cross-validation to get out-of-fold probability predictions for both models. Then we calculate a **weighted average** (XGBoost has 2x weight because it is slightly stronger). Finally, we find the exact probability threshold that maximizes the F1 Score.

In [ ]:
print("Generating cross-validation probabilities...")
probs_xgb = cross_val_predict(xgb, X, y, cv=cv, method='predict_proba')[:, 1]
probs_rf = cross_val_predict(rf, X, y, cv=cv, method='predict_proba')[:, 1]

# Weighted average (2 parts XGBoost, 1 part Random Forest)
ensemble_probs = (2 * probs_xgb + probs_rf) / 3

print("Optimizing threshold for maximum F1...")
best_t = 0.5
best_f1 = 0

for t in np.arange(0.20, 0.65, 0.005):
    preds = (ensemble_probs >= t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_t = t

print(f"\nBest Threshold: {best_t:.3f}")
print(f"Best CV F1 Score: {best_f1:.4f}")

cv_preds = (ensemble_probs >= best_t).astype(int)
print("\nClassification Report (Cross-Validated):")
print(classification_report(y, cv_preds))

### Step 8: Train Final Models & Predict

In [ ]:
# Train models on all available data
xgb.fit(X, y)
rf.fit(X, y)
print("Models trained on all data!")

# Generate test probabilities
test_probs_xgb = xgb.predict_proba(X_test)[:, 1]
test_probs_rf = rf.predict_proba(X_test)[:, 1]

# Apply weights and threshold
final_test_probs = (2 * test_probs_xgb + test_probs_rf) / 3
final_predictions = (final_test_probs >= best_t).astype(int)

print(f"\nFinal Predictions (threshold={best_t:.3f}):")
print(f"  Not Converted (0): {(final_predictions == 0).sum()}")
print(f"  Converted (1): {(final_predictions == 1).sum()}")
print(f"  Predicted conversion rate: {final_predictions.mean():.4f}")

### Step 9: Feature Importance (from XGBoost)

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("Top 15 Most Important Features:")
for feat, imp in importances.head(15).items():
    print(f"  {feat:25s} {imp:.4f}")

### Step 10: Create Submission File

In [ ]:
submission = pd.DataFrame({
    "User_ID": private_test_raw["User_ID"],
    "Converted": final_predictions
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")
print(f"Shape: {submission.shape}")
submission.head(10)

In [ ]:
# Verify submission format matches sample
sample = pd.read_csv("sample_submission.csv")

print("=== Submission Verification ===")
print(f"Columns match: {list(submission.columns) == list(sample.columns)}")
print(f"Row count match: {len(submission) == len(sample)} ({len(submission)} rows)")
print(f"User IDs match: {(submission['User_ID'].values == sample['User_ID'].values).all()}")
print(f"All predictions binary: {submission['Converted'].isin([0, 1]).all()}")
print("\nSubmission is VALID!")